In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('train.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## The Plan — What a Pipeline Solves

The previous notebook (WithoutPipelines) required:
- Managing 6+ separate transformer objects
- Manually tracking column positions
- Fitting each transformer separately (with risk of fitting on test data by mistake)
- Saving multiple pickle files
- Replicating all transforms manually at inference

**A `Pipeline` chains all these steps into a single object.** You call `.fit()` once and `.predict()` once — internally it executes every transformation step in order, handles train-only fitting automatically, and serializes to a single `.pkl` file.

**Tools used here:**
| Class | Purpose |
|---|---|
| `ColumnTransformer` | Apply different transformers to different columns in parallel |
| `Pipeline` / `make_pipeline` | Chain transformers + estimator sequentially |
| `SelectKBest(chi2)` | Automatic feature selection — keeps the k most informative features |
| `GridSearchCV` | Hyperparameter tuning — works seamlessly with Pipeline |

In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

## Train-Test Split

Same split as the non-pipeline version — `random_state=42` ensures identical train/test rows, making the accuracy comparison fair.

In [6]:
# train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [7]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [8]:
y_train.sample(5)

751    1
203    0
152    0
454    0
126    0
Name: Survived, dtype: int64

---
## Building the Transformers

### trf1 — Imputation (`ColumnTransformer`)

`ColumnTransformer` applies **different transformers to different columns simultaneously** and outputs a single combined array.

**Why column indices `[2]` and `[6]` instead of column names?**  
`ColumnTransformer` outputs a numpy array — column names are lost after the first transformation. All subsequent steps in the pipeline receive **arrays, not DataFrames**. So from trf2 onwards, we must reference columns by their positional index in the output array of the previous step.

At trf1's input, `X_train` still has its original column order:
```
[0]=Pclass  [1]=Sex  [2]=Age  [3]=SibSp  [4]=Parch  [5]=Fare  [6]=Embarked
```

`remainder='passthrough'` keeps all other columns (0,1,3,4,5) unchanged in the output — they're just passed through as-is alongside the imputed columns.

**trf1 output column order** (imputed cols come first, passthrough comes after):
```
[0]=Age(imputed)  [1]=Embarked(imputed)  [2]=Pclass  [3]=Sex  [4]=SibSp  [5]=Parch  [6]=Fare
```

In [9]:
# piplines 1st step -> column Transformer
trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('Impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
],remainder = 'passthrough')
# why we put index of clm rather than its name?

### trf2 — One-Hot Encoding (`ColumnTransformer`)

**Why are `Sex` and `Embarked` encoded together in one `OneHotEncoder` here — unlike the previous notebook?**

In the non-pipeline version, `Embarked` had to be imputed first (it had NaN values), and only then could it be OHE'd. This forced two separate encoders.

Here, `trf1` has **already imputed `Embarked`** — by the time `trf2` runs, there are no NaN values in any column. So both `Sex` and `Embarked` can be passed to a single `OneHotEncoder` in one shot.

**What are the column indices `[1, 6]` in trf2?**  
These refer to positions in trf1's **output** array (not the original DataFrame):
```
trf1 output: [0]=Age(imp)  [1]=Embarked(imp)  [2]=Pclass  [3]=Sex  [4]=SibSp  [5]=Parch  [6]=Fare
```
- Index `1` = imputed Embarked  
- Index `6` = Fare? — Actually wait: Sex was at original index 1, which maps to trf1's passthrough output.

After trf1, the passthrough columns (Pclass=0→2, Sex=1→3, SibSp=3→4, Parch=4→5, Fare=5→6) are reindexed in the output. Index `[1,6]` targets Embarked and Fare — confirm by checking trf2's output shape vs. what OHE produces.

`remainder='passthrough'` again passes non-encoded columns through.

In [10]:
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
],remainder='passthrough')
#why we did sex and embarked together here?

### trf3 — Scaling (`ColumnTransformer`)

**Why `slice(0, 10)` — where does 10 come from?**

After trf2's One-Hot Encoding, let's count the columns in trf2's output:
- OHE on `Sex` → 2 columns (female, male)
- OHE on `Embarked` → 3 columns (C, Q, S)  
- Passthrough remaining 5 numeric columns (Pclass, Age, SibSp, Parch, Fare)

Total = 2 + 3 + 5 = **10 columns**

`slice(0, 10)` means "scale all columns from index 0 to 9" — i.e., all 10 columns.  
`MinMaxScaler` is appropriate here because the next step uses `chi2` (chi-squared) for feature selection, which **requires non-negative values** — MinMaxScaler's [0,1] output satisfies this.

In [11]:
# Scaling
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])
#why 0 to 10

### trf4 — Feature Selection (`SelectKBest`)

`SelectKBest(score_func=chi2, k=8)` automatically selects the **8 most informative features** out of 10 based on the chi-squared statistic between each feature and the target.

**Why `chi2` specifically?**  
Chi-squared tests the independence between two categorical/count variables. After MinMaxScaling to [0,1], all features are non-negative — satisfying chi2's requirement. It measures how much each feature's distribution differs across the two target classes (Survived=0 vs 1).

**Why k=8 (not all 10)?**  
Dropping the 2 least informative features can reduce overfitting and noise. The optimal `k` will later be tuned via GridSearchCV.

In [12]:
# Feature Selection
trf4 = SelectKBest(score_func=chi2,k=8)

### trf5 — Classifier

The final step is the `DecisionTreeClassifier`. In a Pipeline, the last step is always the **estimator** — the object that has `.fit()` and `.predict()`. All previous steps are transformers (they have `.fit_transform()`).

In [13]:
trf5 = DecisionTreeClassifier()

---
## Creating the Pipeline

The `Pipeline` constructor below has a critical copy-paste error:
```python
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf1),   
    ('trf3', trf1),   
    ('trf4', trf1),   
    ('trf5', trf1)    
])
```
Every step references `trf1` (the imputer). This means the pipeline would apply imputation 5 times and never encode, scale, select features, or classify. **This cell is intentionally kept to show the mistake — it is immediately fixed in the next cell using `make_pipeline`.**

In [14]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf1),
    ('trf3',trf1),
    ('trf4',trf1),
    ('trf5',trf1)
])
    

### Pipeline Vs make_pipeline

Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

### `make_pipeline` — The Clean Fix

`make_pipeline` is a convenience function that auto-names each step based on the class name, eliminating the manual naming (and the bug above).

```python
make_pipeline(trf1, trf2, trf3, trf4, trf5)
# auto-names: columntransformer-1, columntransformer-2, columntransformer-3, selectkbest, decisiontreeclassifier
```

The entire preprocessing chain — imputation → encoding → scaling → feature selection → classification — is now one object.

In [15]:
pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

## Training — One Call Does Everything

`pipe.fit(X_train, y_train)` internally executes:
1. `trf1.fit_transform(X_train)` → imputes Age and Embarked
2. `trf2.fit_transform(...)` → OHE encodes Sex and Embarked
3. `trf3.fit_transform(...)` → MinMaxScales all 10 columns
4. `trf4.fit_transform(...)` → selects best 8 features using chi2
5. `trf5.fit(...)` → trains the DecisionTree on the final 8 features

**Crucially:** Each step's `fit` only uses the data flowing through the pipeline from `X_train`. No test data is involved — data leakage is structurally impossible.

In [16]:
pipe.fit(X_train,y_train)

,steps,"[('columntransformer-1', ...), ('columntransformer-2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('Impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


---
## Exploring the Fitted Pipeline

After fitting, every step inside the pipeline has learned its parameters. We can inspect them.

### `pipe.named_steps` — Accessing Each Step

`pipe.named_steps` returns a dictionary mapping step names → fitted transformer/estimator objects.

This lets you introspect any step after fitting — useful for debugging, understanding what was learned, or extracting parameters.

In [17]:
# Inspect all steps in the pipeline
pipe.named_steps


{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('Impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x0000026788AEA020>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

### Drilling Into Step Parameters

You can navigate into any step's learned parameters using chained attribute access.

In [18]:
pipe.named_steps['columntransformer-1'].transformers_[0]

('impute_age', SimpleImputer(), [2])

`statistics_` on a fitted `SimpleImputer` shows the fill value it learned from the training data.

**Age imputer learned mean: ~29.5 years** — this is the value that replaced all 177 missing `Age` entries in `X_train`.

In [19]:
pipe.named_steps['columntransformer-1'].transformers_[0][1].statistics_

array([29.49884615])

`Embarked` imputer learned most frequent value: **'S' (Southampton)** — used to fill the 2 missing embarked values.

In [20]:
pipe.named_steps['columntransformer-1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

### Visual Diagram of the Pipeline

`set_config(display='diagram')` enables a visual HTML representation of the pipeline structure in Jupyter — shows each step as a connected block. Useful for quickly verifying the pipeline's structure is correct.

In [21]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

---
## Prediction and Accuracy

`pipe.predict(X_test)` runs `X_test` through all 5 transformation steps automatically before classifying. No manual column manipulation needed.

In [22]:
# Predict
y_pred = pipe.predict(X_test)

In [23]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

**Pipeline accuracy: ~62.6%**

This is lower than the manual approach (~78.2%) — not because pipelines are worse, but because:
1. `SelectKBest(k=8)` is discarding 2 features that may actually be useful
2. The `max_depth` of the Decision Tree is unconstrained (default = full depth = overfitting)

Both of these will be addressed by GridSearchCV below — the pipeline approach makes hyperparameter tuning dramatically easier.

In [24]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6256983240223464

---
## Cross-Validation with Pipeline

`cross_val_score` splits `X_train` into 5 folds, trains on 4 and validates on 1, cycling through all combinations.

**The key advantage:** the entire pipeline — including all fitting steps — is re-run from scratch on each fold. This means the imputer, encoder, scaler, and feature selector are all fit only on the training folds and applied to the validation fold. This gives a **truly unbiased estimate** of generalization performance.

In the manual approach without pipeline, you would have had to manually re-fit and re-apply every transformer for each fold — extremely tedious and error-prone.

In [25]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.6391214419383433)

---
## GridSearchCV with Pipeline

### The Double-Underscore `__` Syntax

To tune a parameter inside a pipeline step, use:
```
'stepname__parametername': [values]
```

Here `'decisiontreeclassifier__max_depth'` tunes the `max_depth` parameter of the `DecisionTreeClassifier` step.

**This is only possible because of Pipeline.** Without pipeline, you'd have to retrain and re-apply all transformers for every hyperparameter combination manually.

GridSearchCV runs the full pipeline (transform + train + validate) for each combination of parameters across all CV folds — everything is handled automatically.

In [26]:
# gridsearchcv
params = {
    'decisiontreeclassifier__max_depth':[1,2,3,4,5,None]
}

In [27]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'decisiontreeclassifier__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('impute_age', ...), ('Impute_embarked', ...)]"


**Best CV score: ~63.9%** with `max_depth=2`

The optimal tree depth is quite shallow — this confirms that the unconstrained tree was overfitting the training data. A shallower tree generalizes better.

You can now call `grid.best_estimator_.predict(X_test)` to predict using the best found pipeline.

In [28]:
grid.best_score_

np.float64(0.6391214419383433)

In [29]:
grid.best_params_

{'decisiontreeclassifier__max_depth': 2}

---
## Exporting the Pipeline — One File vs Three

The entire preprocessing + classification chain is saved as **a single `pipe.pkl` file**.

Compare this to the non-pipeline approach which required saving `ohe_sex.pkl`, `ohe_embarked.pkl`, and `clf.pkl` separately.

**At inference time:** load `pipe.pkl`, call `pipe.predict(raw_input)` — done. No manual transformation steps. No column order juggling. No risk of mismatching components.